# BM25S Baseline for MIRACL Arabic\n
**Modern Python implementation - No Java required**\n
\n
This notebook:\n
1. Downloads MIRACL Arabic corpus\n
2. Indexes with BM25S\n
3. Evaluates on dev queries\n
4. Compares to Pyserini baseline

## 1. Install Dependencies

In [ ]:
!pip install -q bm25s datasets pytrec_eval PyStemmer nltk

## 2. Download MIRACL Arabic Data

In [ ]:
import json
import gzip
import requests
import os
from tqdm import tqdm

# Setup directories
os.makedirs("data/miracl_ar", exist_ok=True)

# Download corpus (5 chunks)
base_url = "https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ar/docs-{}.jsonl.gz"
num_chunks = 5

corpus_docs = []
corpus_ids = []

print("Downloading MIRACL Arabic corpus...")
for i in range(num_chunks):
    file_url = base_url.format(i)
    temp_file = f"data/miracl_ar/docs-{i}.jsonl.gz"
    
    # Download
    print(f"Chunk {i+1}/{num_chunks}...")
    response = requests.get(file_url, stream=True)
    with open(temp_file, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
    
    # Parse
    with gzip.open(temp_file, 'rt', encoding='utf-8') as f:
        for line in f:
            doc = json.loads(line)
            # Combine title + text (same as Pyserini)
            corpus_docs.append(f"{doc['title']} {doc['text']}")
            corpus_ids.append(doc['docid'])
    
    # Cleanup
    os.remove(temp_file)

print(f"✅ Loaded {len(corpus_docs):,} documents")

## 3. Download Queries and Qrels

In [ ]:
# Download dev queries
queries_url = "https://huggingface.co/datasets/miracl/miracl/resolve/main/miracl-v1.0-ar/topics/topics.miracl-v1.0-ar-dev.tsv"
qrels_url = "https://huggingface.co/datasets/miracl/miracl/resolve/main/miracl-v1.0-ar/qrels/qrels.miracl-v1.0-ar-dev.tsv"

# Download queries
response = requests.get(queries_url)
with open("data/miracl_ar/topics.tsv", 'wb') as f:
    f.write(response.content)

# Download qrels
response = requests.get(qrels_url)
with open("data/miracl_ar/qrels.tsv", 'wb') as f:
    f.write(response.content)

# Parse queries
queries = {}
with open("data/miracl_ar/topics.tsv", 'r', encoding='utf-8') as f:
    for line in f:
        qid, query = line.strip().split('\t')
        queries[qid] = query

# Parse qrels
qrels = {}
with open("data/miracl_ar/qrels.tsv", 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        qid, _, docid, rel = parts
        if qid not in qrels:
            qrels[qid] = {}
        qrels[qid][docid] = int(rel)

print(f"✅ Loaded {len(queries)} queries")
print(f"✅ Loaded qrels for {len(qrels)} queries")

## 4. Tokenize Corpus (Arabic-aware)

In [ ]:
import bm25s
import Stemmer
import nltk
from nltk.corpus import stopwords
import pickle

# Download NLTK stopwords
nltk.download('stopwords', quiet=True)

# BM25S comes with a simple tokenizer, but we'll use Arabic stemming
print("Tokenizing corpus (this may take 5-10 minutes)...")

# Create Arabic stemmer
stemmer = Stemmer.Stemmer('arabic')

# Get comprehensive Arabic stopwords from NLTK (245+ words)
arabic_stopwords = stopwords.words('arabic')
print(f"Using {len(arabic_stopwords)} Arabic stopwords from NLTK")

# Tokenize corpus
corpus_tokens = bm25s.tokenize(
    corpus_docs,
    stopwords=arabic_stopwords,
    stemmer=stemmer,
    show_progress=True
)

print("✅ Tokenization complete")

## 5. Build BM25S Index

In [ ]:
print("Building BM25S index...")

# Create retriever with BM25 parameters matching Pyserini
retriever = bm25s.BM25(
    method="lucene",  # Use Lucene-style BM25
    k1=0.9,           # Match your Pyserini config
    b=0.4
)

# Index the corpus
retriever.index(corpus_tokens, show_progress=True)

print("✅ Index built")

# Save index for reuse
retriever.save("data/miracl_ar/bm25s_index")

with open('data/miracl_ar/corpus_ids.pkl', 'wb') as f:
    pickle.dump(corpus_ids, f)
    
print("✅ Index saved to disk")

## 6. Run Retrieval

In [ ]:
print("Running retrieval on dev queries...")

# Tokenize queries
query_texts = [queries[qid] for qid in sorted(queries.keys())]
query_ids = sorted(queries.keys())

query_tokens = bm25s.tokenize(
    query_texts,
    stopwords=arabic_stopwords,
    stemmer=stemmer,
    show_progress=True
)

# Retrieve top 100 documents per query
results, scores = retriever.retrieve(
    query_tokens,
    k=100,
    show_progress=True
)

print("✅ Retrieval complete")

## 7. Convert to TREC Format

In [ ]:
# Create run file in TREC format
os.makedirs("results/bm25s", exist_ok=True)
run_file = "results/bm25s/run.txt"

with open(run_file, 'w') as f:
    for i, qid in enumerate(query_ids):
        doc_indices = results[i]
        doc_scores = scores[i]
        
        for rank, (doc_idx, score) in enumerate(zip(doc_indices, doc_scores)):
            docid = corpus_ids[doc_idx]
            f.write(f"{qid} Q0 {docid} {rank+1} {score:.5f} bm25s\n")

print(f"✅ Run file saved: {run_file}")

## 8. Evaluate Results

In [ ]:
import pytrec_eval

# Load run file
with open(run_file, 'r') as f:
    run_data = pytrec_eval.parse_run(f)

# Evaluate
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels,
    {'recall_10', 'recall_100', 'ndcg_cut_10', 'recip_rank'}
)

results_eval = evaluator.evaluate(run_data)

# Aggregate scores
metrics = ['recall_10', 'recall_100', 'ndcg_cut_10', 'recip_rank']
aggs = {m: 0.0 for m in metrics}

for qid in results_eval:
    for m in metrics:
        aggs[m] += results_eval[qid][m]

num_queries = len(results_eval)

print("\n" + "="*50)
print("🎯 BM25S RESULTS ON MIRACL ARABIC")
print("="*50)
print(f"Recall@100:  {aggs['recall_100']/num_queries:.4f}  (Target: ~0.889)")
print(f"NDCG@10:     {aggs['ndcg_cut_10']/num_queries:.4f}  (Target: ~0.481)")
print(f"Recall@10:   {aggs['recall_10']/num_queries:.4f}  (Thesis metric)")
print(f"MRR:         {aggs['recip_rank']/num_queries:.4f}")
print("="*50)

## 9. Comparison Summary

**Expected Results:**
- Official Pyserini (Java 11): Recall@100 ≈ 0.889
- Your Pyserini (Java 21): Recall@100 ≈ 0.79
- BM25S (this notebook): Recall@100 ≈ ?

**Key Advantages of BM25S:**
1. ✅ No Java dependency
2. ✅ Pure Python (easy to modify)
3. ✅ Fast indexing and retrieval
4. ✅ Easy QE integration
5. ✅ Modern, maintained codebase

## 10. Test Query Enhancement Integration

In [ ]:
# Example: How easy it is to add QE layer
def dummy_query_enhancement(query):
    """Placeholder for your future LLM-based QE"""
    # This is where you'll add: HyDE, Query2Doc, etc.
    return query + " معلومات"  # Just adds "information" in Arabic

# Test on first query
test_qid = query_ids[0]
original_query = queries[test_qid]
enhanced_query = dummy_query_enhancement(original_query)

print(f"Original: {original_query}")
print(f"Enhanced: {enhanced_query}")

# Retrieve with enhanced query
enhanced_tokens = bm25s.tokenize(
    [enhanced_query],
    stopwords=arabic_stopwords,
    stemmer=stemmer
)

enhanced_results, enhanced_scores = retriever.retrieve(enhanced_tokens, k=10)

print(f"\n✅ QE integration works! Retrieved {len(enhanced_results[0])} docs")
print("\nThis is where your thesis contribution will happen! 🚀")

## 11. Loading Saved Index (For Future Sessions)

**Use this when you want to skip indexing and load a pre-built index**

In [ ]:
# ============================================
# OPTION A: Load Index + Corpus IDs
# ============================================
import bm25s
import Stemmer
import nltk
from nltk.corpus import stopwords
import pickle

print("Loading saved index...")

# Load the BM25S index
retriever = bm25s.BM25.load("data/miracl_ar/bm25s_index", load_corpus=True)

# Load corpus IDs (you need to save these separately)
# Add this to cell 5 after building index:
# with open('data/miracl_ar/corpus_ids.pkl', 'wb') as f:
#     pickle.dump(corpus_ids, f)

with open('data/miracl_ar/corpus_ids.pkl', 'rb') as f:
    corpus_ids = pickle.load(f)

# Setup stemmer and stopwords (needed for query tokenization)
nltk.download('stopwords', quiet=True)
stemmer = Stemmer.Stemmer('arabic')
arabic_stopwords = stopwords.words('arabic')

print(f"✅ Index loaded with {len(corpus_ids):,} documents")

In [ ]:
# ============================================
# OPTION B: Quick Search Example
# ============================================

# Test query
test_query = "ما هي عاصمة مصر"  # What is the capital of Egypt?

# Tokenize query
query_tokens = bm25s.tokenize(
    [test_query],
    stopwords=arabic_stopwords,
    stemmer=stemmer
)

# Retrieve top 10 documents
results, scores = retriever.retrieve(query_tokens, k=10)

print(f"Query: {test_query}")
print(f"\nTop 10 Results:")
print("="*50)
for rank, (doc_idx, score) in enumerate(zip(results[0], scores[0]), 1):
    docid = corpus_ids[doc_idx]
    print(f"{rank}. DocID: {docid} | Score: {score:.4f}")